## Multipatch manipulation

Assembly of several patchs sharing control points and degrees of freedom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (BSpline, BSplineSurface, ControlPointManager,
    Patch, HRefiner, SubdivisionRefiner, PRefiner, BezierExtractor,
    GlobalDOFManager, PatchDOFManager, PatchAssembly
)
from yeti_iga.future.plotting import plot_patches_2d

In [ ]:
# Define a set of control points
mgr = ControlPointManager(dim=2)

mgr.add_point([0.0, 1.0])    # CP 0
mgr.add_point([1.0, 1.0])    # CP 1
mgr.add_point([1.0, 0.0])    # CP 2
mgr.add_point([0.0, 2.0])    # CP 3
mgr.add_point([2.0, 2.0])    # CP 4
mgr.add_point([2.0, 0.0])    # CP 5
mgr.add_point([1.0, -1.0])    # CP 6
mgr.add_point([0.0, -1.0])    # CP 7
mgr.add_point([2.0, -2.0])    # CP 8
mgr.add_point([0.0, -2.0])    # CP 9

# Plot points with their indices in CP manager
def plot_control_points(coords):
    fig, ax = plt.subplots()

    ax.scatter(coords[:, 0], coords[:, 1], zorder=3)
    for i, (x, y) in enumerate(coords):
        ax.annotate(str(i), (x, y), textcoords="offset points", xytext=(6, 6))

    ax.set_aspect('equal')
    ax.grid(True)
    plt.show()

coords = mgr.coords_view()  # shape (n_points, 2)
plot_control_points(coords)




In [ ]:
# define a set of degrees of freedom

dofs_per_control_point = [2] * mgr.n_points
global_dof_manager = GlobalDOFManager(dofs_per_control_point)

In [ ]:
# create 2 patchs sharing control points
# Patch 1
su1 = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sv1 = BSpline(1, np.array([0., 0., 1., 1.]))
surf1 = BSplineSurface(su1, sv1)
mapping1 = np.array([0, 1, 2, 3, 4, 5])
patch_dof_manager1 = PatchDOFManager(
    dofs_per_control_point=2,
    control_points=mapping1,
    global_dof_manager=global_dof_manager
)
patch1 = Patch(surf1, mgr, mapping1.tolist(), [3, 2], patch_dof_manager1)

# Patch 2
sv2 = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
su2 = BSpline(1, np.array([0., 0., 1., 1.]))
surf2 = BSplineSurface(su2, sv2)
mapping2 = np.array([7, 9, 6, 8, 2, 5])
patch_dof_manager2 = PatchDOFManager(
    dofs_per_control_point=2,
    control_points=mapping2,
    global_dof_manager=global_dof_manager
)
patch2 = Patch(surf2, mgr, mapping2.tolist(), [2, 3], patch_dof_manager2)

# Assembly
assembly = PatchAssembly()
assembly.add_patch(patch1)
assembly.add_patch(patch2)

# Detect shared control points
assembly.detect_shared_control_points()
shared_map = assembly.get_shared_control_points_map()
print('Shared control points:')
for cp_idx, patch_indices in shared_map.items():
    print(f"Global ID {cp_idx} shared by patchs {patch_indices}")

In [ ]:
# Visualize actual patch geometry (iso-parametric borders + CPs), one color per patch
plot_patches_2d(assembly, show_control_points=True, show_control_point_indices=True,
                 title="Initial assembly")

## Safe refinement of a shared patch

`patch1` and `patch2` share their `ControlPointManager` (CPs 2 and 5 are common to both). Refining `patch1` without precaution would rewrite the whole shared CP pool with only `patch1`'s points, **corrupting `patch2`**.

`refine_1d` now accepts a `protected_global_ids` set: CPs in this set are never recomputed or renumbered — they keep the id and coordinates already present in the pool. All other CPs of `patch1` (its private CPs) get a fresh, dense block of ids, in u-fastest order.

This set is obtained directly from `assembly.get_shared_control_points_map()`.

In [ ]:
# Control points shared with patch2 -- never to be touched by patch1's refinement
protected_ids = set(shared_map.keys())
print("Protected (shared) global ids:", protected_ids)

# Snapshot patch2's control points before refining patch1
patch2_cps_before = [list(patch2.control_point(i)) for i in range(patch2.n_cp)]
print("patch2 control points (before):", patch2_cps_before)
print("mgr.n_points (before):", mgr.n_points)

In [ ]:
# Refine patch1 along direction 0 (u), protecting the CPs shared with patch2
T_1d = SubdivisionRefiner(direction=0, n_levels=1).refine_1d(patch1, protected_ids)

print("patch1.n_cp (after):", patch1.n_cp)
print("mgr.n_points (after):", mgr.n_points)

In [ ]:
# patch2 must be completely unaffected
patch2_cps_after = [list(patch2.control_point(i)) for i in range(patch2.n_cp)]
print("patch2 control points (after):", patch2_cps_after)
print("patch2 unchanged:", patch2_cps_before == patch2_cps_after)

# The shared CPs (2 and 5) must still be detected as shared, with the same ids
assembly.detect_shared_control_points()
print("Shared control points (after):", assembly.get_shared_control_points_map())

coords = mgr.coords_view()
plot_control_points(coords)

In [ ]:
plot_patches_2d(assembly, show_control_points=True, show_control_point_indices=True,
                 title="After transverse (u) refinement of patch1")

## Propagating refinement across a shared edge

Refining `patch1` along direction 0 (u) above was a *transverse* refinement: the shared edge doesn't grow, only the count of CPs across it does, so the two protected corner CPs stay exactly where they are.

Refining *along* the shared edge is different. In this notebook the edge is parametrized by direction 1 (v) on `patch1`'s side but by direction 0 (u) on `patch2`'s side — a *crossed* interface (`varying_direction_a=1`, `varying_direction_b=0`). Subdividing `patch1` along v alone would create a new CP in the middle of the shared edge that `patch2` knows nothing about, producing a non-conforming mesh — and refining `patch2` along the wrong direction wouldn't grow the right edge either.

`assembly.detect_interfaces()` finds this edge (direction, side, and orientation on each side) from the already-detected shared CPs. `assembly.refine_with_propagation(patch_index, direction, refine_1d_fn)` then:
1. refines the requested patch along `direction`,
2. automatically refines every neighbor connected through a matching interface along *its own* corresponding direction (which the crossed case shows is generally NOT the same `direction` value),
3. merges the new edge CPs created independently on both sides (they are coordinate-identical, since both come from the same deterministic algorithm applied to the same original boundary).

`refine_1d_fn(patch, direction, protected_global_ids)` is any callable that refines `patch` in-place along `direction` while respecting `protected_global_ids` — typically a thin wrapper around `SubdivisionRefiner`/`PRefiner`'s `refine_1d`. `direction` is passed explicitly to the callback (rather than baked into a closure) precisely because it can differ between the two sides of a crossed interface.

In [ ]:
assembly.detect_interfaces()
print("Detected interfaces:", assembly.get_interfaces())

# Any callable that refines `patch` in-place along `direction`, respecting
# protected_global_ids. `direction` is passed explicitly because on a
# crossed interface (u of one patch glued to v of the other) the neighbor
# must be refined along its OWN matching direction, not necessarily the
# direction given to refine_with_propagation() below.
def refine_1d_fn(patch, direction, protected_global_ids):
    SubdivisionRefiner(direction=direction, n_levels=1).refine_1d(patch, protected_global_ids)

patch1_n_cp_before, patch2_n_cp_before = patch1.n_cp, patch2.n_cp

assembly.refine_with_propagation(patch_index=0, direction=1, refine_1d_fn=refine_1d_fn)

print(f"patch1.n_cp: {patch1_n_cp_before} -> {patch1.n_cp}")
print(f"patch2.n_cp: {patch2_n_cp_before} -> {patch2.n_cp}")

# The shared edge must now have 3 control points: the 2 original corners
# plus the new midpoint, merged into a single shared id on both sides.
print("Shared control points (after propagation):", assembly.get_shared_control_points_map())

coords = mgr.coords_view()
plot_control_points(coords)

In [ ]:
# The shared edge must line up exactly between the two colors, with the new
# merged midpoint shown only once (not duplicated between patch colors).
plot_patches_2d(assembly, show_control_points=True, show_control_point_indices=True,
                 title="After propagated refinement (merged shared edge)")

## Growing and rebuilding the DOF managers after propagation

`refine_with_propagation()` above only touches control points -- it never updates any `PatchDOFManager`. The new midpoint CP merged on the shared edge has no dofs yet, and `global_dof_manager` doesn't even know it exists.

`assembly.update_dof_managers(global_dof_manager, dofs_per_cp)`:
1. grows `global_dof_manager` to cover every control point now referenced by the assembly (assigning `dofs_per_cp` fresh dofs to each newly covered CP),
2. rebuilds each patch's `PatchDOFManager` from its CURRENT `global_indices`.

Because `GlobalDOFManager` maps dofs by control-point id, the merged boundary CP automatically gets the **same** global dofs on both `patch1` and `patch2` -- no explicit tracking of which CPs were merged is needed.

Call this **after** `refine_with_propagation()` and **before** `compact()`: `compact()` renumbers control point ids, which would desynchronize this cp-id-indexed mapping if dofs were assigned beforehand against soon-to-be-stale ids.</cell>


In [ ]:
assembly.update_dof_managers(global_dof_manager, dofs_per_cp=2)

print("global_dof_manager.n_control_points():", global_dof_manager.n_control_points())

# Every control point shared between patch1 and patch2 must now resolve to
# the exact same global dofs, regardless of which patch's local position
# we look it up through.
shared_map = assembly.get_shared_control_points_map()
for cp_id, patch_indices in shared_map.items():
    expected = global_dof_manager.get_dof_indices(cp_id)
    print(f"cp_id={cp_id} shared by patches {patch_indices}, global dofs={expected}")
    for pidx in patch_indices:
        patch = patch1 if pidx == 0 else patch2
        local_pos = patch.global_indices.index(cp_id)
        got = patch.dof_manager.get_global_dof_indices(local_pos)
        assert got == expected, f"Mismatch for patch {pidx}, cp {cp_id}: {got} != {expected}"
print("All shared control points have matching dofs across patches: OK")

## Reclaiming orphaned control points

Look closely at the plot above: some CPs appear duplicated at the same location (e.g. the original corners of `patch1` that are *not* shared with `patch2`). This is expected: `refine_1d` relocates every *private* CP of `patch1` to a fresh dense block, even when its coordinates didn't change — the old slot is simply abandoned (orphaned), not reused, to keep the operation safe and cheap for shared patches.

This is a deliberate trade-off: refinement itself never deletes anything (multiple patches might still be reading the pool concurrently), so cleanup is a separate, explicit step. Call `assembly.compact()` once you're done with **all** the refinements you need — not after every single step — to rewrite the shared CP pool without the orphaned entries.

`compact()` keeps DOF numbering untouched (a `PatchDOFManager` maps *local position* → *global dof*, independent of CP ids), but it invalidates any previously detected shared-CP map, so `detect_shared_control_points()` must be called again afterwards.

In [ ]:
# Snapshot both patches right before compact() -- compaction must not move
# anything geometrically, only renumber/reclaim ids.
patch1_cps_before_compact = [list(patch1.control_point(i)) for i in range(patch1.n_cp)]
patch2_cps_before_compact = [list(patch2.control_point(i)) for i in range(patch2.n_cp)]
print("mgr.n_points (before compact):", mgr.n_points)

assembly.compact()

print("mgr.n_points (after compact):", mgr.n_points)

# Both patches must still be geometrically correct after compaction
patch1_cps_after_compact = [list(patch1.control_point(i)) for i in range(patch1.n_cp)]
patch2_cps_after_compact = [list(patch2.control_point(i)) for i in range(patch2.n_cp)]
print("patch1 control points:", patch1_cps_after_compact)
print("patch2 control points:", patch2_cps_after_compact)
print("patch1 unchanged by compact:", patch1_cps_before_compact == patch1_cps_after_compact)
print("patch2 unchanged by compact:", patch2_cps_before_compact == patch2_cps_after_compact)

# Old shared-CP ids are no longer valid -- re-detect them
assembly.detect_shared_control_points()
print("Shared control points (after compact):", assembly.get_shared_control_points_map())

coords = mgr.coords_view()
plot_control_points(coords)

In [ ]:
plot_patches_2d(assembly, show_control_points=True, show_control_point_indices=True,
                 title="Final assembly (compacted)")